In [26]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import logit, expit
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, average_precision_score
import pandas as pd
from torch.utils.data import Dataset, DataLoader

from pytorch_grad_cam import GradCAM, LayerCAM, HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image


from torchvision import transforms
from PIL import Image


import random
import yaml
from pathlib import Path
import time
import datetime
from string import Template
import os
import json
#from ..dataset.abstract_dataset import DeepfakeAbstractBaseDataset

from detectors import DETECTOR


#from metrics.utils import get_test_metrics

In [4]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)
    
def init_seed(config):
    if config['manualSeed'] is None:
        config['manualSeed'] = random.randint(1, 10000)
    random.seed(config['manualSeed'])
    torch.manual_seed(config['manualSeed'])
    if config['cuda']:
        torch.cuda.manual_seed_all(config['manualSeed'])

In [5]:
def prepare_testing_data(config):


    transformations = transforms.Compose([
        transforms.Resize((config["resolution"], config["resolution"])), 
        transforms.ToTensor(),
        transforms.Normalize(mean=config["mean"], std=config["std"])
    ])

    json_path = f"{config["dataset_json_folder"]}/{config["test_dataset"][0]}.json"
    
    dataset = DeepfakeFrameDataset(
        json_path=json_path, 
        split="test", 
        transform=transformations
    )

    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

    return dataloader

In [6]:
def init(config_path, weights_path, test_dataset, config_test):
        config = load_config(config_path)
        test_config = load_config(config_test)

        config.update(test_config)
        config["test_dataset"] = test_dataset
        config["weights_path"] = weights_path

        if config['cudnn']:
                cudnn.benchmark = True

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        init_seed(config)


        test_data_loaders = prepare_testing_data(config)

        model_class = DETECTOR[config['model_name']]
        model = model_class(config).to(device)

        ckpt = torch.load(config["weights_path"], map_location=device)

        if 'state_dict' in ckpt:
                ckpt = ckpt['state_dict']

        new_weights = {}

        for key, value in ckpt.items():
                new_key = key.replace('module.', '')
                if 'base_model.' in new_key:
                        new_key = new_key.replace('base_model.', 'backbone.')
                if 'classifier.' in new_key:
                        new_key = new_key.replace('classifier.', 'head.')
                if 'HRNet_layer.' in new_key:
                        new_key = new_key.replace('HRNet_layer.', 'backbone.')
                new_weights[new_key] = value

        if type(model).__name__ == "EffortDetector":
                model.load_state_dict(new_weights, strict=False)
        else:
                model.load_state_dict(new_weights, strict=True)
        print('===> Load checkpoint done!')

        return test_data_loaders, model, device, config


In [7]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.float32):
            return obj.item()
        if isinstance(obj, np.int64):
            return obj.item()
        return super().default(obj)

In [8]:
class DeepfakeFrameDataset(Dataset):
    def __init__(self, json_path, split="test", transform=None):
        """
        Args:
            json_path (str): Chemin vers le fichier JSON.
            split (str): 'train' ou 'test' selon les données que vous voulez charger.
            transform (callable, optional): Transformations torchvision à appliquer.
        """
        self.transform = transform
        self.frames_data = [] # Va contenir notre liste aplatie
        
        # Dictionnaire pour convertir vos labels texte en entiers (pour PyTorch)
        # À adapter si vos classes s'appellent autrement !
        self.label_map = {
            "ConfDF_real": 0,
            "ConfDF_fake": 1
        }

        # 1. Chargement du fichier JSON
        with open(json_path, 'r') as f:
            data = json.load(f)
            
        # 2. Navigation dans l'arborescence du JSON
        # On récupère la racine (ici "ConfDF_norm_v2_th")
        root_key = list(data.keys())[0]
        dataset_root = data[root_key]
        
        # On parcourt les différentes classes (ex: ConfDF_real, ConfDF_fake...)
        for class_name, class_data in dataset_root.items():
            
            # On vérifie si le dossier de split ('train' ou 'test') existe
            if split in class_data:
                split_data = class_data[split]
                
                # On parcourt chaque vidéo de ce split
                for video_name, video_info in split_data.items():
                    # On récupère le label et on le convertit en chiffre (0 ou 1)
                    label_str = video_info.get("label", class_name)
                    label_idx = self.label_map.get(label_str, -1) 
                    
                    # On récupère la liste des chemins des frames
                    frames_list = video_info.get("frames", [])
                    
                    # On ajoute CHAQUE frame à notre liste aplatie
                    for frame_path in frames_list:
                        self.frames_data.append({
                            "frame_path": frame_path,
                            "label": label_idx,
                            "video_name": video_name
                        })

        print(f"Dataset initialisé : {len(self.frames_data)} frames trouvées pour le split '{split}'.")

    def __len__(self):
        # PyTorch a besoin de savoir combien d'images il y a au total
        return len(self.frames_data)

    def __getitem__(self, idx):
        # PyTorch demande l'élément à l'index 'idx'
        item = self.frames_data[idx]
        frame_path = item["frame_path"]
        label = item["label"]
        video_name = item["video_name"]

        # 3. Chargement de l'image
        try:
            # On utilise PIL pour être parfaitement compatible avec torchvision.transforms
            image = Image.open(frame_path).convert('RGB')
        except Exception as e:
            print(f"Erreur de chargement pour l'image {frame_path} : {e}")
            raise e

        # 4. Application des transformations (redimensionnement, normalisation en tenseurs)
        if self.transform:
            image = self.transform(image)

        # On renvoie le triplet (Tenseur Image, Label, Nom de la vidéo)
        return image, label, video_name


In [9]:
def grad_cam_from_dataset(model, test_dataset_loader, device, cam, target_layers):
    

    model.eval()
    targets = [ClassifierOutputTarget(1)]


    for batch in test_dataset_loader:
        print(batch)
        break
        grayscale_cam = cam(input_tensor=image["tensor"], targets=targets)
    
    # grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    # grayscale_cam = grayscale_cam[0, :]
    # visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

In [18]:
def find_last_spatial_conv_layer(model):
    """
    Parcourt le modèle et renvoie la dernière couche Conv2d 
    qui a un noyau spatial (ex: 3x3). Cela évite les couches 
    de projection 1x1 qui perdent souvent leurs gradients.
    """
    last_conv_layer = None
    for name, module in model.named_modules():
        # On cherche un Conv2d avec un kernel de (3, 3)
        if isinstance(module, nn.Conv2d) and module.kernel_size == (3, 3):
            last_conv_layer = module
            
    return last_conv_layer

# Entry Point

In [11]:
exps = pd.read_csv("exps_exai.csv")

CONFIG_TEST = "config/test_config.yaml"
CONFIG_DETECTORS = "config/detector/"
WEIGHTS_BASE = "weights/"
EXPS_DIR = "exps/"

model_name = None
config_detector_path = None
weights_path = None
test_dataset = None

predictions_nps = None
label_nps = None
images = None
elapsed_time = None
config = None

for _, row in exps.iterrows():
    model_name = row['model_name']
    config_detector_path = row['config']
    weights_path = row ['weights']
    test_dataset = row['test_dataset']
    if row["done"] == True:
        print(f"Already done {model_name} ({weights_path}) on {test_dataset}. Skip")
        continue
    print(f"Start {model_name} ({weights_path} on {test_dataset})")


    test_data_loaders, model, device, config = init(CONFIG_DETECTORS + config_detector_path, WEIGHTS_BASE + weights_path, [test_dataset], CONFIG_TEST)
    target_layers = [find_last_conv_layer(model)]
    cam = GradCAM(model=model, target_layers=target_layers)

    start = time.time()

    #predictions_nps, label_nps, images, logits_nps = grad_cam_from_dataset(model, test_data_loaders, device, cam, target_layers)

    stop = time.time()
    elapsed_time = stop-start
    #save_display_metrics(predictions_nps, label_nps, logits_nps, images, model_name, test_dataset, weights_path, elapsed_time, config, EXPS_DIR)

Already done clip_base (train_on_df40-all-ff/clip.pth) on ConfDF_norm_v2_all. Skip
Already done i3d (train_on_df40-all-ff/i3d.pth) on ConfDF_norm_v2_all. Skip
Already done spsl (train_on_ff-orig/spsl_best.pth) on ConfDF_norm_v2_all. Skip
Start xception (train_on_df40-all-ff/xception.pth on ConfDF_norm_v2_all)
Dataset initialisé : 24986 frames trouvées pour le split 'test'.
===> Load checkpoint done!
Couche cible trouvée : Conv2d(2048, 512, kernel_size=(1, 1), stride=(1, 1))


In [12]:
import cv2
import json
import numpy as np
import torch

def recuperer_frames_json(json_path, nom_video):
    """Cherche une vidéo dans le JSON et renvoie la liste triée de ses frames."""
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    root_key = list(data.keys())[0]
    dataset_root = data[root_key]
    
    for class_name, class_data in dataset_root.items():
        for split in ["train", "test", "val"]:
            if split in class_data and nom_video in class_data[split]:
                video_info = class_data[split][nom_video]
                frames = video_info.get("frames", [])
                label = video_info.get("label", class_name)
                
                frames.sort() # Indispensable pour l'ordre temporel
                return frames, label
                
    return None, None


def creer_video_gradcam_visages(json_path, nom_video, output_path, model, target_layers, fps=30, model_input_size=(256, 256)):
    """
    Génère une vidéo Grad-CAM à partir de frames de visages recadrées (ex: 256x256).
    """

    torch.set_grad_enabled(True)
    frames_paths, label_texte = recuperer_frames_json(json_path, nom_video)
    
    if not frames_paths:
        print(f"Erreur : La vidéo {nom_video} n'a pas été trouvée.")
        return
        
    print(f"Création de la vidéo ({label_texte}) : {len(frames_paths)} frames...")

    
    # 1. Lire la première image pour récupérer sa taille exacte (normalement 256x256)
    premiere_image = cv2.imread(frames_paths[0])
    hauteur_originale, largeur_originale, _ = premiere_image.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (largeur_originale, hauteur_originale))

    # Forcer l'activation des gradients sur tout le modèle
    for param in model.parameters():
        param.requires_grad = True
        
    # On emballe votre modèle pour qu'il comprenne Grad-CAM
    modele_adapte = WrapperPourGradCam(model)
    
    
    # 3. Initialiser Grad-CAM avec le modèle ADAPTÉ (mais on garde les mêmes target_layers !)
    cam = GradCAM(model=modele_adapte, target_layers=target_layers)
    # -------------------------------
    
    # Adaptez la classe cible selon votre configuration (1 = Fake, 0 = Real)
    target_class = 0 if label_texte == "ConfDF_real" else 1 
    targets = [ClassifierOutputTarget(target_class)]
    device = next(model.parameters()).device
    
    for i, frame_path in enumerate(frames_paths):
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
            
        # Image d'origine pour la visualisation finale (ex: 256x256)
        rgb_img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb_img_float = np.float32(rgb_img) / 255.0
        
        # Redimensionnement POUR LE MODÈLE uniquement (si nécessaire)
        # Si votre Xception prend du 256x256, ça ne changera rien. 
        # S'il prend du 299x299, ça adaptera l'image pour l'inférence.
        img_pour_modele = cv2.resize(rgb_img_float, model_input_size)
        
        # Transformation en tenseur PyTorch
        input_tensor = preprocess_image(img_pour_modele,
                                        mean=[0.5, 0.5, 0.5], # Valeurs de votre DataLoader
                                        std=[0.5, 0.5, 0.5])
        input_tensor = input_tensor.to(device)
        
        # --- GÉNÉRATION GRAD-CAM ---
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        
        # Nettoyage des dimensions inutiles (squeeze) 
        # Cela transforme (1, 1, 256, 256) en (256, 256) directement
        grayscale_cam = np.squeeze(grayscale_cam)
        
        # On redimensionne la heatmap pour qu'elle corresponde à votre image d'origine
        grayscale_cam_resized = cv2.resize(grayscale_cam, (largeur_originale, hauteur_originale))
        
        # On superpose la heatmap sur le visage recadré
        cam_image = show_cam_on_image(rgb_img_float, grayscale_cam_resized, use_rgb=True)
        
        # Conversion pour l'enregistrement vidéo
        cam_image_bgr = cv2.cvtColor(np.uint8(255 * cam_image), cv2.COLOR_RGB2BGR)
        out.write(cam_image_bgr)
        
        if (i + 1) % 30 == 0:
            print(f"Traitement : {i + 1}/{len(frames_paths)} frames...")

    out.release()
    print(f"✅ Vidéo sauvegardée sous : {output_path}")

In [13]:
class WrapperPourGradCam(nn.Module):
    def __init__(self, modele_original):
        super().__init__()
        self.modele_original = modele_original
        # On force le mode évaluation (important pour Dropout/BatchNorm)
        self.modele_original.eval() 
        
    def forward(self, x):
        # On s'assure que le tenseur d'entrée accepte de recevoir des gradients
        # C'est souvent l'astuce qui débloque Grad-CAM !
            
        data_dict = {'image': x}
        output = self.modele_original(data_dict)
        
        res = None
        if isinstance(output, dict):
            for cle in ['prob', 'cls', 'logits', 'out']:
                if cle in output:
                    res = output[cle]
                    break
            if res is None:
                res = list(output.values())[0]
        else:
            res = output

        # GESTION DES DIMENSIONS
        if res.dim() == 0:
            res = res.unsqueeze(0)
        if res.dim() == 1:
            res = res.unsqueeze(0)
            
        # CAS BINAIRE
        if res.size(-1) == 1:
            res_real = 1.0 - res 
            res = torch.cat([res_real, res], dim=-1)

        return res

In [ ]:
def find_mid_spatial_conv_layer(model, recul=3):
    """
    Récupère une couche de convolution un peu plus "haute" dans le réseau 
    pour avoir une meilleure résolution spatiale.
    recul=1 : dernière couche (blob)
    recul=3 ou 4 : couches intermédiaires (plus de détails)
    """
    conv_layers = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.kernel_size == (3, 3):
            conv_layers.append(module)
            
    # S'il y a assez de couches, on prend celle demandée en partant de la fin
    if len(conv_layers) >= recul:
        return conv_layers[-recul]
    elif len(conv_layers) > 0:
        return conv_layers[-1] # Sécurité
    return None

In [42]:
# =====================================
# UTILISATION
# =====================================
if __name__ == "__main__":
    nom_video = "bruce_willis_1.mp4"
    chemin_json = "datasets_json/ConfDF_norm_v2_all.json"

    couche_cible = find_mid_spatial_conv_layer(model)
    print(f"Couche cible retenue pour Grad-CAM : {couche_cible}")
    
    creer_video_gradcam_visages2(
        json_path=chemin_json,
        nom_video=nom_video,
        output_path="visage_gradcam-fake.mp4",
        model=model,
        target_layers=[couche_cible],
        fps=30,
        model_input_size=(256, 256) # Mettez (299, 299) si votre Xception l'exige
    )

Couche cible retenue pour Grad-CAM : Conv2d(728, 728, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=728, bias=False)
Création de la vidéo (ConfDF_fake) : 300 frames...
Traitement : 30/300 frames...
Traitement : 60/300 frames...
Traitement : 90/300 frames...
Traitement : 120/300 frames...
Traitement : 150/300 frames...
Traitement : 180/300 frames...
Traitement : 210/300 frames...
Traitement : 240/300 frames...
Traitement : 270/300 frames...
Traitement : 300/300 frames...
✅ Vidéo sauvegardée sous : visage_gradcam-fake.mp4


In [40]:
import cv2
import json
import numpy as np
import torch
import torch.nn as nn
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image

class WrapperPourGradCam2(nn.Module):
    def __init__(self, modele_original):
        super().__init__()
        self.modele_original = modele_original
        self.modele_original.eval() 
        
    def forward(self, x):
        # !! ATTENTION !! Ne JAMAIS faire x.requires_grad_(True) ici.
        # Grad-CAM s'en charge en amont.
            
        data_dict = {'image': x}
        output = self.modele_original(data_dict)
        
        res = None
        if isinstance(output, dict):
            for cle in ['prob', 'cls', 'logits', 'out']:
                if cle in output:
                    res = output[cle]
                    break
            if res is None:
                res = list(output.values())[0]
        else:
            res = output

        # GESTION DES DIMENSIONS (sans casser le graphe de calcul)
        if res.dim() == 0:
            res = res.unsqueeze(0)
        if res.dim() == 1:
            res = res.unsqueeze(0)
            
        # CAS BINAIRE : on recrée le format [Score_Real, Score_Fake]
        if res.size(-1) == 1:
            res_real = 1.0 - res 
            res = torch.cat([res_real, res], dim=-1)

        return res


def creer_video_gradcam_visages2(json_path, nom_video, output_path, model, target_layers, fps=30, model_input_size=(256, 256)):
    """
    Génère une vidéo Grad-CAM à partir de frames de visages recadrées.
    """
    # 0. SÉCURITÉ PYTORCH : On force l'activation des gradients globalement
    torch.set_grad_enabled(True)
    
    frames_paths, label_texte = recuperer_frames_json(json_path, nom_video)
    
    if not frames_paths:
        print(f"Erreur : La vidéo {nom_video} n'a pas été trouvée.")
        return
        
    print(f"Création de la vidéo ({label_texte}) : {len(frames_paths)} frames...")

    premiere_image = cv2.imread(frames_paths[0])
    hauteur_originale, largeur_originale, _ = premiere_image.shape

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (largeur_originale, hauteur_originale))

    # Forcer l'activation des gradients sur les poids du modèle
    model.eval()
    for param in model.parameters():
        param.requires_grad = True
        
    # On emballe le modèle
    modele_adapte = WrapperPourGradCam2(model)
    
    # Initialisation Grad-CAM
    cam = LayerCAM(model=modele_adapte, target_layers=target_layers)
    
    #cam = HiResCAM(model=modele_adapte, target_layers=target_layers)

    target_class = 0 if label_texte == "ConfDF_real" else 1 
    targets = [ClassifierOutputTarget(target_class)]
    device = next(model.parameters()).device
    
    for i, frame_path in enumerate(frames_paths):
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
            
        rgb_img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb_img_float = np.float32(rgb_img) / 255.0
        
        img_pour_modele = cv2.resize(rgb_img_float, model_input_size)
        
        input_tensor = preprocess_image(img_pour_modele,
                                        mean=[0.5, 0.5, 0.5],
                                        std=[0.5, 0.5, 0.5])
        input_tensor = input_tensor.to(device)
        
        # --- GÉNÉRATION GRAD-CAM ---
        # Grad-CAM s'occupera d'activer requires_grad sur input_tensor
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        
        grayscale_cam = np.squeeze(grayscale_cam)
        grayscale_cam_resized = cv2.resize(grayscale_cam, (largeur_originale, hauteur_originale))
        
        cam_image = show_cam_on_image(rgb_img_float, grayscale_cam_resized, use_rgb=True)
        cam_image_bgr = cv2.cvtColor(np.uint8(255 * cam_image), cv2.COLOR_RGB2BGR)
        out.write(cam_image_bgr)
        
        if (i + 1) % 30 == 0:
            print(f"Traitement : {i + 1}/{len(frames_paths)} frames...")

    out.release()
    print(f"✅ Vidéo sauvegardée sous : {output_path}")